In [ ]:
import pandas as pd
df_employee = pd.read_csv("/content/sample_data/employee_record.csv")

print(df_employee)

   EmployeeID  Age       Department          JobRole  TenureYears  \
0     EMP1001   60  Human Resources          Analyst           15   
1     EMP1002   50          Finance        Developer            2   
2     EMP1003   36  Human Resources          Manager            3   
3     EMP1004   29  Human Resources    HR Specialist            2   
4     EMP1005   42            Sales    HR Specialist           15   
..        ...  ...              ...              ...          ...   
95    EMP1096   55  Human Resources    HR Specialist            1   
96    EMP1097   54            Sales  Sales Executive           15   
97    EMP1098   44            Sales        Developer            6   
98    EMP1099   45          Finance    HR Specialist           14   
99    EMP1100   58  Human Resources        Developer            5   

   CompensationBand  MonthlyIncome  PerformanceRating  JobSatisfaction  \
0                 C          76293                  1                2   
1                 A    

In [ ]:
#attrition by rate
overall_attrition = (
    df_employee["ExitStatus"].eq("Exited").mean() * 100
)

print(f"Attrition Rate: {overall_attrition:.1f}%")

Attrition Rate: 15.0%


In [ ]:
#attrition by department
dept_attrition = pd.crosstab(
    df_employee["Department"],
    df_employee["ExitStatus"],
    normalize="index"
) * 100

print(dept_attrition.round(1))

ExitStatus              Active  Exited
Department                            
Finance                   90.9     9.1
Human Resources           82.6    17.4
IT                        81.2    18.8
Research & Development    78.6    21.4
Sales                     88.0    12.0


In [ ]:
#Attrition by tenure
df_employee["TenureBand"] = pd.cut(
    df_employee["TenureYears"],
    bins=[-1, 2, 5, 10, float("inf")],
    labels=["0-2 years", "3-5 years", "6-10 years", "11+ years"]
)

tenure_attrition = pd.crosstab(
    df_employee["TenureBand"],
    df_employee["ExitStatus"],
    normalize="index"
) * 100

print(tenure_attrition.round(1))

ExitStatus  Active  Exited
TenureBand                
0-2 years     87.0    13.0
3-5 years     92.3     7.7
6-10 years    92.6     7.4
11+ years     75.7    24.3


In [ ]:
#Attrition by role
role_attrition = (
    df_employee.groupby("JobRole")
      .agg(
          Employees=("EmployeeID", "count"),
          Exited=("ExitStatus", lambda x: (x == "Exited").sum())
      )
)

role_attrition["AttritionRate"] = (
    role_attrition["Exited"] /
    role_attrition["Employees"] * 100
).round(1)

print(role_attrition.sort_values("AttritionRate", ascending=False))

                    Employees  Exited  AttritionRate
JobRole                                             
Research Scientist         10       3           30.0
HR Specialist              20       4           20.0
Sales Executive            21       4           19.0
Analyst                    18       2           11.1
Developer                  10       1           10.0
Manager                    21       1            4.8


In [ ]:
#Attrition by compensation band
comp_attrition = pd.crosstab(
    df_employee["CompensationBand"],
    df_employee["ExitStatus"],
    normalize="index"
) * 100

print(comp_attrition.round(1))

ExitStatus        Active  Exited
CompensationBand                
A                   71.4    28.6
B                   85.2    14.8
C                   90.5     9.5
D                   95.8     4.2


In [ ]:
#Early-Tenure Attrition
df_employee["TenureMonths"] = (df_employee["TenureYears"] * 12).round().astype(int)
df_employee["Early_Tenure"] = df_employee["TenureMonths"].apply(
    lambda x: "Yes" if x < 12 else "No"
)
#df_employee["Early_Tenure"] = df_employee["Tenure"].apply(
#    lambda x: "Yes" if x < 1 else "No"
#)

In [ ]:
#early_tenure_attrition = (
early_tenure_attrition = (
    df_employee[df_employee["Early_Tenure"] == "Yes"]["ExitStatus"]
    .eq("Exited")
    .mean() * 100
)

print("Early-Tenure Attrition Rate:",
      round(early_tenure_attrition, 2), "%")

Early-Tenure Attrition Rate: 12.5 %


In [ ]:
#Monthly Attrition Rate
early_tenure_attrition = (
    df_employee[df_employee["Early_Tenure"] == "Yes"]["ExitStatus"]
    .eq("Exited")
    .mean() * 100
)

print("Early-Tenure Attrition Rate:",
      round(early_tenure_attrition, 2), "%")

Early-Tenure Attrition Rate: 12.5 %


In [ ]:
df_employee["JoiningDate"] = pd.to_datetime(
    df_employee["JoiningDate"],
    errors="coerce"
)

df_employee["ExitDate"] = pd.to_datetime(
    df_employee["ExitDate"],
    errors="coerce"
)

In [ ]:
#Annual Attrition Rate
print(df_employee.columns.tolist())
##   df_employee.groupby("Year")["ExitStatus"]
#    .apply(lambda x: (x == "Exited").mean() * 100)
#    .reset_index(name="Annual_Attrition_Rate")
#    )
# Convert ExitDate to datetime
df_employee["ExitDate"] = pd.to_datetime(df_employee["ExitDate"], errors="coerce")

# Create Year from ExitDate
df_employee["Year"] = df_employee["ExitDate"].dt.year

# Calculate annual attrition rate
annual_attrition = (
    df_employee.dropna(subset=["Year"])
    .groupby("Year")["ExitStatus"]
    .apply(lambda x: (x == "Exited").mean() * 100)
    .reset_index(name="Annual_Attrition_Rate")
)

print(annual_attrition)

['EmployeeID', 'Age', 'Department', 'JobRole', 'TenureYears', 'CompensationBand', 'MonthlyIncome', 'PerformanceRating', 'JobSatisfaction', 'WorkLifeBalance', 'Overtime', 'ExitStatus', 'TenureBand', 'TenureMonths', 'Early_Tenure', 'JoiningDate', 'ExitDate']
     Year  Annual_Attrition_Rate
0  2025.0                  100.0
1  2026.0                  100.0


In [ ]:
# Early tenure indicator
df_employee["Early_Tenure"] = (
    df_employee["TenureYears"]
    .apply(lambda x: "Yes" if x < 1 else "No")
)

# Exit indicator
df_employee["Exit_Flag"] = (
    df_employee["ExitStatus"]
    .apply(lambda x: 1 if x == "Exited" else 0)
)
print(df_employee)

   EmployeeID  Age       Department          JobRole  TenureYears  \
0     EMP1001   60  Human Resources          Analyst           15   
1     EMP1002   50          Finance        Developer            2   
2     EMP1003   36  Human Resources          Manager            3   
3     EMP1004   29  Human Resources    HR Specialist            2   
4     EMP1005   42            Sales    HR Specialist           15   
..        ...  ...              ...              ...          ...   
95    EMP1096   55  Human Resources    HR Specialist            1   
96    EMP1097   54            Sales  Sales Executive           15   
97    EMP1098   44            Sales        Developer            6   
98    EMP1099   45          Finance    HR Specialist           14   
99    EMP1100   58  Human Resources        Developer            5   

   CompensationBand  MonthlyIncome  PerformanceRating  JobSatisfaction  \
0                 C          76293                  1                2   
1                 A    

In [ ]:
start_date = df_employee["JoiningDate"].min().to_period("M").start_time
end_date = max(
    df_employee["JoiningDate"].max(),
    df_employee["ExitDate"].dropna().max()
).to_period("M").end_time

months = pd.period_range(
    start=start_date,
    end=end_date,
    freq="M"
)

monthly_results = []

for month in months:

    month_start = month.start_time
    month_end = month.end_time

    # Employees active at beginning of month
    beginning_headcount = (
        (df_employee["JoiningDate"] < month_start) &
        (
            df_employee["ExitDate"].isna() |
            (df_employee["ExitDate"] >= month_start)
        )
    ).sum()

    # Employees who joined during month
    joiners = (
        (df_employee["JoiningDate"] >= month_start) &
        (df_employee["JoiningDate"] <= month_end)
    ).sum()

    # Employees who exited during month
    exits = (
        (df_employee["ExitDate"] >= month_start) &
        (df_employee["ExitDate"] <= month_end)
    ).sum()

    # Employees active at end of month
    ending_headcount = (
        (df_employee["JoiningDate"] <= month_end) &
        (
            df_employee["ExitDate"].isna() |
            (df_employee["ExitDate"] > month_end)
        )
    ).sum()

    # Average headcount
    average_headcount = (
        beginning_headcount + ending_headcount
    ) / 2

    # Attrition rate
    if average_headcount > 0:
        attrition_rate = (
            exits / average_headcount
        ) * 100
    else:
        attrition_rate = 0

    monthly_results.append({
        "Month": str(month),
        "Beginning_Headcount": beginning_headcount,
        "Joiners": joiners,
        "Exits": exits,
        "Ending_Headcount": ending_headcount,
        "Average_Headcount": round(average_headcount, 2),
        "Monthly_Attrition_Rate": round(attrition_rate, 2)
    })


monthly_attrition = pd.DataFrame(monthly_results)


# --------------------------------------------------
# 3. Annual Attrition Rate
# --------------------------------------------------

years = range(
    int(df_employee["JoiningDate"].dt.year.min()),
    int(max(
        df_employee["JoiningDate"].dt.year.max(),
        df_employee["ExitDate"].dropna().dt.year.max()
    )) + 1
)

annual_results = []

for year in years:

    year_start = pd.Timestamp(f"{year}-01-01")
    year_end = pd.Timestamp(f"{year}-12-31")

    # Beginning headcount
    beginning_headcount = (
        (df_employee["JoiningDate"] < year_start) &
        (
            df_employee["ExitDate"].isna() |
            (df_employee["ExitDate"] >= year_start)
        )
    ).sum()

    # Joiners
    joiners = (
        (df_employee["JoiningDate"] >= year_start) &
        (df_employee["JoiningDate"] <= year_end)
    ).sum()

    # Exits
    exits = (
        (df_employee["ExitDate"] >= year_start) &
        (df_employee["ExitDate"] <= year_end)
    ).sum()

    # Ending headcount
    ending_headcount = (
        (df_employee["JoiningDate"] <= year_end) &
        (
            df_employee["ExitDate"].isna() |
            (df_employee["ExitDate"] > year_end)
        )
    ).sum()

    # Average headcount
    average_headcount = (
        beginning_headcount + ending_headcount
    ) / 2

    # Annual attrition rate
    if average_headcount > 0:
        attrition_rate = (
            exits / average_headcount
        ) * 100
    else:
        attrition_rate = 0

    annual_results.append({
        "Year": year,
        "Beginning_Headcount": beginning_headcount,
        "Joiners": joiners,
        "Exits": exits,
        "Ending_Headcount": ending_headcount,
        "Average_Headcount": round(average_headcount, 2),
        "Annual_Attrition_Rate": round(attrition_rate, 2)
    })


annual_attrition = pd.DataFrame(annual_results)


# --------------------------------------------------
# 4. Display results
# --------------------------------------------------

print("\nMONTHLY ATTRITION")
print(monthly_attrition.to_string(index=False))

print("\nANNUAL ATTRITION")
print(annual_attrition.to_string(index=False))


# --------------------------------------------------
# 5. Save results
# --------------------------------------------------

monthly_attrition.to_csv(
    "monthly_attrition_rates.csv",
    index=False
)

annual_attrition.to_csv(
    "annual_attrition_rates.csv",
    index=False
)


MONTHLY ATTRITION
  Month  Beginning_Headcount  Joiners  Exits  Ending_Headcount  Average_Headcount  Monthly_Attrition_Rate
2010-10                    0        1      0                 1                0.5                    0.00
2010-11                    1        0      0                 1                1.0                    0.00
2010-12                    1        1      0                 2                1.5                    0.00
2011-01                    2        1      0                 3                2.5                    0.00
2011-02                    3        2      0                 5                4.0                    0.00
2011-03                    5        1      0                 6                5.5                    0.00
2011-04                    6        1      0                 7                6.5                    0.00
2011-05                    7        2      0                 9                8.0                    0.00
2011-06                    

In [ ]:
from scipy.stats import chi2_contingency

categorical_variables = [
    "Department",
    "JobRole",
    "CompensationBand",
    "Overtime",
    "TenureBand",
    "Early_Tenure",
    "JobSatisfaction",
    "WorkLifeBalance"
]

chi_square_results = []

for variable in categorical_variables:

    if variable not in df_employee.columns:
        continue

    contingency_table = pd.crosstab(
        df_employee[variable],
        df_employee["ExitStatus"]
    )

    chi2, p_value, dof, expected = chi2_contingency(
        contingency_table
    )

    chi_square_results.append({
        "Variable": variable,
        "Chi_Square": round(chi2, 3),
        "P_Value": round(p_value, 4),
        "Significant": "Yes" if p_value < 0.05 else "No"
    })


chi_square_results = pd.DataFrame(chi_square_results)

print("\n========================================")
print("CHI-SQUARE TEST RESULTS")
print("========================================")

print(
    chi_square_results
    .sort_values("P_Value")
    .to_string(index=False)
)


# ============================================================
# 4. T-TESTS
#    Compare numerical variables between Active and Exited
# ============================================================

numerical_variables = [
    "Age",
    "TenureYears",
    "TenureMonths",
    "MonthlyIncome",
    "PerformanceRating",
    "JobSatisfaction",
    "WorkLifeBalance"
]

ttest_results = []

for variable in numerical_variables:

    if variable not in df_employee.columns:
        continue

    active = df_employee.loc[
        df_employee["Attrition"] == 0,
        variable
    ].dropna()

    exited = df_employee.loc[
        df_employee["Attrition"] == 1,
        variable
    ].dropna()

    t_stat, p_value = ttest_ind(
        active,
        exited,
        equal_var=False
    )

    ttest_results.append({
        "Variable": variable,
        "Active_Mean": round(active.mean(), 2),
        "Exited_Mean": round(exited.mean(), 2),
        "T_Statistic": round(t_stat, 3),
        "P_Value": round(p_value, 4),
        "Significant": "Yes" if p_value < 0.05 else "No"
    })


ttest_results = pd.DataFrame(ttest_results)

print("\n========================================")
print("T-TEST RESULTS")
print("========================================")

print(
    ttest_results
    .sort_values("P_Value")
    .to_string(index=False)
)


# ============================================================
# 5. SIMPLE LOGISTIC REGRESSION
# ============================================================
#
# Predict Attrition using individual numerical variables.
#
# Logistic regression is appropriate because Attrition
# is binary: 0 = Active, 1 = Exited.
# ============================================================

regression_variables = [
    "Age",
    "TenureYears",
    "MonthlyIncome",
    "PerformanceRating",
    "JobSatisfaction",
    "WorkLifeBalance"
]

regression_data = df_employee[
    ["Attrition"] + regression_variables
].dropna()

X = regression_data[regression_variables]
y = regression_data["Attrition"]

# Add intercept
X = sm.add_constant(X)

logistic_model = sm.Logit(y, X).fit(disp=False)

print("\n========================================")
print("LOGISTIC REGRESSION")
print("========================================")

print(logistic_model.summary())


# ============================================================
# 6. ODDS RATIOS
# ============================================================

odds_ratios = pd.DataFrame({
    "Variable": logistic_model.params.index,
    "Coefficient": logistic_model.params.values,
    "Odds_Ratio": np.exp(logistic_model.params.values),
    "P_Value": logistic_model.pvalues.values
})

odds_ratios["Significant"] = np.where(
    odds_ratios["P_Value"] < 0.05,
    "Yes",
    "No"
)

print("\n========================================")
print("ODDS RATIOS")
print("========================================")

print(
    odds_ratios
    .round(4)
    .to_string(index=False)
)


# ============================================================
# 7. SAVE RESULTS
# ============================================================

#chi_square_results.to_csv(
#    "chi_square_results.csv",
#    index=False
#)

#ttest_results.to_csv(
#    "ttest_results.csv",
#    index=False
#)

#odds_ratios.to_csv(
#    "logistic_regression_results.csv",
#    index=False
#)

#print("\nResults saved:")
#print(" - chi_square_results.csv")
#print(" - ttest_results.csv")
#print(" - logistic_regression_results.csv")



CHI-SQUARE TEST RESULTS
        Variable  Chi_Square  P_Value Significant
 JobSatisfaction      10.784   0.0130         Yes
CompensationBand       6.749   0.0804          No
      TenureBand       4.357   0.2254          No
         JobRole       4.563   0.4715          No
 WorkLifeBalance       1.512   0.6796          No
      Department       1.512   0.8244          No
        Overtime       0.028   0.8665          No
    Early_Tenure       0.000   1.0000          No


KeyError: 'Attrition'